# NeuroScan - Combined Training (IAM + Dyslexia)
## Automatic Dataset Download with KaggleHub

This notebook automatically downloads:
- **IAM Handwriting Database** → Normal writing samples
- **Dyslexia Handwriting Dataset** → Reversal patterns

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Download trained model at the end

In [ ]:
# Cell 1: Install dependencies
!pip install tensorflow opencv-python-headless scikit-learn seaborn kagglehub -q
print("✅ Dependencies installed!")

In [ ]:
# Cell 2: Check GPU
import tensorflow as tf
import numpy as np
import cv2
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
import glob
import random

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU: {gpus[0]}")
else:
    print("⚠️ No GPU! Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 3: Download IAM Handwriting Database
import kagglehub

print("📥 Downloading IAM Handwriting Database...")
iam_path = kagglehub.dataset_download("nibinv23/iam-handwriting-word-database")
print(f"✅ IAM downloaded to: {iam_path}")

# Show contents
print("\nContents:")
for item in os.listdir(iam_path)[:10]:
    print(f"  {item}")

In [ ]:
# Cell 4: Download Dyslexia Handwriting Dataset
print("📥 Downloading Dyslexia Handwriting Dataset...")
dyslexia_path = kagglehub.dataset_download("drizasazanitaisa/dyslexia-handwriting-dataset")
print(f"✅ Dyslexia downloaded to: {dyslexia_path}")

# Check if RAR needs extraction
rar_files = glob.glob(f"{dyslexia_path}/**/*.rar", recursive=True)
if rar_files:
    print(f"\n📦 Found RAR file: {rar_files[0]}")
    !apt-get install unrar -qq
    for rar in rar_files:
        print(f"Extracting {rar}...")
        !unrar x -pWanAsy321 "{rar}" "{dyslexia_path}/" -o+
    print("✅ Extracted!")

# Show contents
print("\nContents:")
!find "{dyslexia_path}" -type d | head -15

In [ ]:
# Cell 5: Explore both datasets
print("=" * 60)
print("DATASET EXPLORATION")
print("=" * 60)

# Find all images
iam_images = glob.glob(f"{iam_path}/**/*.png", recursive=True)
dyslexia_images = glob.glob(f"{dyslexia_path}/**/*.png", recursive=True)

print(f"\n📊 IAM images: {len(iam_images)}")
print(f"📊 Dyslexia images: {len(dyslexia_images)}")

# Sample paths
if iam_images:
    print(f"\nIAM sample: {iam_images[0]}")
if dyslexia_images:
    print(f"Dyslexia sample: {dyslexia_images[0]}")

In [ ]:
# Cell 6: Categorize Dyslexia dataset folders
from collections import defaultdict

# Group dyslexia images by folder
dyslexia_folders = defaultdict(list)
for img in dyslexia_images:
    parent = Path(img).parent.name
    dyslexia_folders[parent].append(img)

print("\n=== Dyslexia Dataset Folders ===")
for folder, imgs in sorted(dyslexia_folders.items(), key=lambda x: -len(x[1])):
    print(f"  {folder}: {len(imgs)} images")

# Identify reversal vs normal folders
REVERSAL_KEYWORDS = ['reversal', 'dyslexia', 'corrected']
NORMAL_KEYWORDS = ['normal', 'healthy']

reversal_images = []
normal_dyslexia_images = []

for folder, imgs in dyslexia_folders.items():
    folder_lower = folder.lower()
    if any(kw in folder_lower for kw in REVERSAL_KEYWORDS):
        reversal_images.extend(imgs)
    elif any(kw in folder_lower for kw in NORMAL_KEYWORDS):
        normal_dyslexia_images.extend(imgs)

print(f"\n📊 Reversal images: {len(reversal_images)}")
print(f"📊 Normal (from dyslexia set): {len(normal_dyslexia_images)}")

In [ ]:
# Cell 7: Configuration
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 50

# Limit samples for balanced training
MAX_SAMPLES_PER_CLASS = 5000

CLASS_NAMES = ['Normal', 'Dyslexia']
NUM_CLASSES = 2

print(f"✅ Config: IMG={IMG_SIZE}, BATCH={BATCH_SIZE}, MAX={MAX_SAMPLES_PER_CLASS}")

In [ ]:
# Cell 8: Image loading function

def load_image(path, size=IMG_SIZE):
    """Load and preprocess image."""
    try:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        img = cv2.resize(img, (size, size))
        img = img.astype(np.float32) / 255.0
        return np.expand_dims(img, axis=-1)
    except:
        return None

def load_images_from_list(image_list, max_samples, label_name):
    """Load images from a list of paths."""
    images = []
    random.shuffle(image_list)
    
    for i, img_path in enumerate(image_list):
        if len(images) >= max_samples:
            break
        img = load_image(img_path)
        if img is not None:
            images.append(img)
        
        if (i+1) % 1000 == 0:
            print(f"  {label_name}: Loaded {len(images)} images...")
    
    return images

print("✅ Loading functions ready")

In [ ]:
# Cell 9: Load NORMAL class (IAM + Normal from Dyslexia set)
print("=== Loading NORMAL Class ===")

# Combine IAM and normal dyslexia images for NORMAL class
all_normal_sources = iam_images + normal_dyslexia_images
random.seed(42)
random.shuffle(all_normal_sources)

print(f"Total normal sources: {len(all_normal_sources)}")

X_normal = load_images_from_list(all_normal_sources, MAX_SAMPLES_PER_CLASS, "NORMAL")
X_normal = np.array(X_normal)
print(f"\n✅ Loaded {len(X_normal)} NORMAL images")

In [ ]:
# Cell 10: Load DYSLEXIA class (Reversals + Synthetic)
print("=== Loading DYSLEXIA Class ===")

X_dyslexia = []

# Load reversal images from dyslexia dataset
if reversal_images:
    print(f"Loading {len(reversal_images)} reversal images...")
    X_dyslexia = load_images_from_list(reversal_images, MAX_SAMPLES_PER_CLASS, "REVERSAL")
    print(f"  Loaded {len(X_dyslexia)} reversal images")

# If not enough, create synthetic dyslexia samples
if len(X_dyslexia) < len(X_normal):
    print(f"\n📝 Creating synthetic dyslexia samples...")
    
    def create_synthetic_dyslexia(img):
        """Apply dyslexia-like transformations."""
        result = img.copy()
        transform = random.choice(['hflip', 'vflip', 'distort', 'rotate'])
        
        if transform == 'hflip':
            result = cv2.flip(result, 1)  # Horizontal flip (b->d)
        elif transform == 'vflip':
            result = cv2.flip(result, 0)  # Vertical flip (p->d)
        elif transform == 'rotate':
            h, w = result.shape[:2]
            angle = random.uniform(-20, 20)
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
            result = cv2.warpAffine(result, M, (w, h), borderValue=1.0)
        elif transform == 'distort':
            # Add noise
            noise = np.random.normal(0, 0.1, result.shape)
            result = np.clip(result + noise, 0, 1)
        
        return result.astype(np.float32)
    
    # Create synthetic samples from normal images
    needed = len(X_normal) - len(X_dyslexia)
    print(f"  Creating {needed} synthetic samples...")
    
    for i in range(needed):
        src_img = X_normal[i % len(X_normal)]
        synthetic = create_synthetic_dyslexia(src_img)
        X_dyslexia.append(synthetic)
        
        if (i+1) % 1000 == 0:
            print(f"    Created {i+1} synthetic samples...")

X_dyslexia = np.array(X_dyslexia)
print(f"\n✅ Total DYSLEXIA samples: {len(X_dyslexia)}")

In [ ]:
# Cell 11: Combine and balance dataset
print("=== Combining Dataset ===")

# Balance classes
min_samples = min(len(X_normal), len(X_dyslexia))
print(f"Balancing: {len(X_normal)} normal, {len(X_dyslexia)} dyslexia -> {min_samples} each")

np.random.seed(42)
normal_idx = np.random.choice(len(X_normal), min_samples, replace=False)
dyslexia_idx = np.random.choice(len(X_dyslexia), min_samples, replace=False)

# Create labels
y_normal = np.zeros(min_samples, dtype=np.int32)
y_dyslexia = np.ones(min_samples, dtype=np.int32)

# Combine
X = np.concatenate([X_normal[normal_idx], X_dyslexia[dyslexia_idx]])
y = np.concatenate([y_normal, y_dyslexia])

# Shuffle
shuffle_idx = np.random.permutation(len(X))
X = X[shuffle_idx]
y = y[shuffle_idx]

print(f"\n✅ Final: {len(X)} samples (Normal: {np.sum(y==0)}, Dyslexia: {np.sum(y==1)})")

In [ ]:
# Cell 12: Visualize samples
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle('Dataset Samples', fontsize=14)

# Normal
normal_idx = np.where(y == 0)[0][:6]
for i, idx in enumerate(normal_idx):
    axes[0, i].imshow(X[idx].squeeze(), cmap='gray')
    axes[0, i].set_title('NORMAL', color='green')
    axes[0, i].axis('off')

# Dyslexia
dyslexia_idx = np.where(y == 1)[0][:6]
for i, idx in enumerate(dyslexia_idx):
    axes[1, i].imshow(X[idx].squeeze(), cmap='gray')
    axes[1, i].set_title('DYSLEXIA', color='red')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Split data
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# Cell 14: Build model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print("✅ Model built!")
model.summary()

In [ ]:
# Cell 15: Callbacks & augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    fill_mode='constant',
    cval=1.0
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', monitor='val_auc', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

print("✅ Ready!")

In [ ]:
# Cell 16: TRAIN
print("="*60)
print("🚀 TRAINING DYSLEXIA DETECTION MODEL")
print(f"   Train: {len(X_train)}, Val: {len(X_val)}")
print("="*60)

train_gen = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)

history = model.fit(
    train_gen,
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

In [ ]:
# Cell 17: Plot history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history.history['auc'], label='Train')
axes[2].plot(history.history['val_auc'], label='Val')
axes[2].set_title('AUC')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# Cell 18: Evaluate
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

test_loss, test_acc, test_auc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"✅ Test AUC: {test_auc:.4f}")

In [ ]:
# Cell 19: Save model
os.makedirs('trained_models', exist_ok=True)

model.save('trained_models/dyslexia_model.keras')
model.save('trained_models/dyslexia_model.h5')

config = {
    'IMG_SIZE': IMG_SIZE,
    'NUM_CLASSES': 2,
    'CLASS_NAMES': CLASS_NAMES,
    'model_type': 'binary-classification',
    'test_accuracy': float(test_acc),
    'test_auc': float(test_auc),
    'training_data': 'IAM Handwriting + Kaggle Dyslexia',
    'total_samples': len(X),
}

with open('trained_models/config.json', 'w') as f:
    json.dump(config, f, indent=2)

import shutil
shutil.copy('training_history.png', 'trained_models/')
shutil.copy('confusion_matrix.png', 'trained_models/')

print("\n✅ Model saved!")
for f in os.listdir('trained_models'):
    size = os.path.getsize(f'trained_models/{f}') / 1024
    unit = 'KB' if size < 1024 else 'MB'
    size = size if size < 1024 else size / 1024
    print(f"  📄 {f} ({size:.1f} {unit})")

In [ ]:
# Cell 20: Download
!zip -r neuroscan_combined_model.zip trained_models/

from google.colab import files
files.download('neuroscan_combined_model.zip')

print("""
========================================
🎉 TRAINING COMPLETE!
========================================

📥 Download: neuroscan_combined_model.zip
📁 Extract to: neurascan-python-ai/models/
🚀 Deploy to Render!

Your model is trained on:
- IAM Handwriting Database (Normal)
- Dyslexia Dataset (Reversal patterns)
""")

In [ ]:
# Cell 21: Test prediction

def predict_image(img_array):
    """Predict dyslexia risk for an image."""
    if len(img_array.shape) == 3:
        img_array = np.expand_dims(img_array, 0)
    
    prob = model.predict(img_array, verbose=0)[0][0]
    score = prob * 100
    
    if score >= 70:
        risk = 'HIGH'
    elif score >= 40:
        risk = 'MODERATE'
    else:
        risk = 'LOW'
    
    return {'score': round(score, 1), 'risk': risk, 'class': 'Dyslexia' if score >= 50 else 'Normal'}

# Test on random samples
print("=== Sample Predictions ===")
for i in range(5):
    idx = np.random.randint(len(X_test))
    result = predict_image(X_test[idx])
    actual = CLASS_NAMES[y_test[idx]]
    print(f"  Sample {i+1}: Score={result['score']:.1f}%, Risk={result['risk']}, Predicted={result['class']}, Actual={actual}")